In [2]:
"""
<벡터 데이터베이스(Vector Stores 또는 Vector Database라고 칭함)>
 * 랭체인 텍스트 분할시 임베딩 처리에 따른 시간이 오래 걸립니다.
 * 이러한 처리를 매번하지 않고, 최초에 처리시 임베딩 결과를 메모리에 저장해 놓고,
   -> 향후 사용시 저장된 임베딩 결과를 불러들여서 효율적으로 사용하면 좋음
 * -> 문자를 숫자로 바꿔놓으면 속도가 빠름.
 * 임베딩된 행렬 데이터 여러개를 저장하는 방법으로 "벡터 데이터베이스"를 사용하고 있음
  - 벡터 데이터베이스를 "벡터 저장소"라고도 칭함


<벡터 데이터베이스 or 벡터 저장소(Vector Stores) 종류>
 * 주로 간편하게 사용 가능한 벡터 데이터베이스로는 크로마(Chroma), 파이스(FAISS)가 있음
   - 오픈소스 기반의 벡터 데이터베이스로,
   - 고차원 벡터 데이터를 빠르고 효율적으로 저장 및 검색 가능한 기능을 제공함
   - 자체 시스템에 관리가 가능함(파일로 저장되어 관리됨)


 * 이외에 밀버스(Milvus), 파이콘(Pinecone), 쿼드런트(Qdrant), 엘라스틱서치(ElasticSearch),
   위비에이트(Weaviate) 등의 벡터 데이터베이스가 존재함


<라이브러리 설치>
 * 크로마(Chroma) 설치 (실치되어 있음)
   >uv pip install langchain_chroma


    ==>> 설치 중에 "protobuf!=4.21.0" ERROR가 발생하면 아래 다시 설치 필요
      (원인) 위 langchain_chroma 설치 중 기존의 protobuf=4.25.8 버전보다
             상위 버전으로 업그레이드 되어서 나타남
      (해결방법) pip install protobuf==4.25.8 버전(이전에 설치되어 있던 버전)으로 설치
          ==> 이후 설치 중 나타나는 오류는 무시해도 무관함...


 * 파이스(FAISS) 설치
   - CPU 버전 설치
     >uv pip install faiss-cpu
     
   - GPU 버전 설치
     (파이썬 3.10까지만 가능, 3.11은 라이브러리 지원이 잘 안되어 cpu 버전으로 설치하여 사용)
     >uv pip install faiss-gpu
"""


'\n<벡터 데이터베이스(Vector Stores 또는 Vector Database라고 칭함)>\n * 랭체인 텍스트 분할시 임베딩 처리에 따른 시간이 오래 걸립니다.\n * 이러한 처리를 매번하지 않고, 최초에 처리시 임베딩 결과를 메모리에 저장해 놓고,\n   -> 향후 사용시 저장된 임베딩 결과를 불러들여서 효율적으로 사용하면 좋음\n * -> 문자를 숫자로 바꿔놓으면 속도가 빠름.\n * 임베딩된 행렬 데이터 여러개를 저장하는 방법으로 "벡터 데이터베이스"를 사용하고 있음\n  - 벡터 데이터베이스를 "벡터 저장소"라고도 칭함\n\n\n<벡터 데이터베이스 or 벡터 저장소(Vector Stores) 종류>\n * 주로 간편하게 사용 가능한 벡터 데이터베이스로는 크로마(Chroma), 파이스(FAISS)가 있음\n   - 오픈소스 기반의 벡터 데이터베이스로,\n   - 고차원 벡터 데이터를 빠르고 효율적으로 저장 및 검색 가능한 기능을 제공함\n   - 자체 시스템에 관리가 가능함(파일로 저장되어 관리됨)\n\n\n * 이외에 밀버스(Milvus), 파이콘(Pinecone), 쿼드런트(Qdrant), 엘라스틱서치(ElasticSearch),\n   위비에이트(Weaviate) 등의 벡터 데이터베이스가 존재함\n\n\n<라이브러리 설치>\n * 크로마(Chroma) 설치 (실치되어 있음)\n   >uv pip install langchain_chroma\n\n\n    ==>> 설치 중에 "protobuf!=4.21.0" ERROR가 발생하면 아래 다시 설치 필요\n      (원인) 위 langchain_chroma 설치 중 기존의 protobuf=4.25.8 버전보다\n             상위 버전으로 업그레이드 되어서 나타남\n      (해결방법) pip install protobuf==4.25.8 버전(이전에 설치되어 있던 버전)으로 설치\n          ==> 이후 설치 중 나타나는 오류는 무시해도 무관함...\n\n\n * 파이스(FAI

## Chroma DB 사용

#### 사용할 라이브러리

In [3]:
### PDF 문서 로더
from langchain_community.document_loaders import PyPDFLoader

### 임베딩 처리 모델(문자를 숫자로 변환 처리)
from langchain_ollama.embeddings import OllamaEmbeddings

### 텍스트 분할기 : 재귀적 문자 텍스트 분할기 사용(다른 것 사용해도 됨)
from langchain_text_splitters import RecursiveCharacterTextSplitter

### 크로마 DB
from langchain_chroma import Chroma

/tmp/ipykernel_21526/3348330508.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### PDF 문서 로드하기

In [4]:
### 사용할 문서 : data 폴더 내에 있는 pdf 파일
# - pdf 문서 로드하기
# - 재귀적 텍스트 분할기 클래스 생성하기 : 청크 분할 기준 1000자 기준, 문맥 연속성 중복 기준은 200자 기준
# - 분할기 이용하여 분할하기 : 분할 후 변수명은 texts로...
# - 출력 : 청크(문서) 갯수 출력(print)

In [5]:
loader = PyPDFLoader("./data/2024_KB_부동산_보고서_최종.pdf")
loader

pages = loader.load()
pages

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
)

texts = text_splitter.split_documents(pages)
print(f"청크 갯수 : {len(texts)}")

청크 갯수 : 135


### 크로마(Chroma) DB에 저장하기

In [6]:
"""
<처리 순서>
  1. 임베딩 클래스 정의(OllamaEmbeddings) : 사용하는 모델은 "nomic-embed-text"
  2. 크로마 DB에 파일 저장 위치 지정
  3. 크로마 DB 클래스 생성 및 저장 처리 : 1번과 2번

크로마 DB 클래스 내부에서 -> 1번(임베딩 처리) 및 2번(저장 처리) 작업이 수행 됩니다.
"""

'\n<처리 순서>\n  1. 임베딩 클래스 정의(OllamaEmbeddings) : 사용하는 모델은 "nomic-embed-text"\n  2. 크로마 DB에 파일 저장 위치 지정\n  3. 크로마 DB 클래스 생성 및 저장 처리 : 1번과 2번\n\n크로마 DB 클래스 내부에서 -> 1번(임베딩 처리) 및 2번(저장 처리) 작업이 수행 됩니다.\n'

In [7]:
# 1. 임베딩 클래스 정의(OllamaEmbeddings) : 사용하는 모델은 "nomic-embed-text"
# - nomic-embed-text 모델 : Nomic AI 기업에서 개발한 고성능 오픈소스 텍스트 임베딩 모델
embedding_function = OllamaEmbeddings(model="nomic-embed-text")
embedding_function

# 2. 크로마 DB에 파일 저장 위치 지정 : 디렉토리 이름만 정의(파일은 자동 생성됨)
# - 저장되는 실제 파일명 : chroma_sqlite3
chroma_save_path = "./Chroma_DB/"

# 3. 크로마 DB 클래스 생성 및 저장 처리 : 1번과 2번
vecotrdb = Chroma.from_documents(
    ### 사용 원본 데이터(로드된 청크(문서) 데이터)
    documents = texts,

    ### 문자를 숫자로 변환하는 변환기(임베딩 전용 모델)
    embedding = embedding_function,

    ### 저장 폴더 위치
    persist_directory = chroma_save_path
)

print("===================[Chroma DB 저장 처리 완료]==================")

===================[Chroma DB 저장 처리 완료]==================


In [8]:
### 저장된 문서의 갯수 확인
# - PDF 분할된 청크(문서)의 갯수와 동일해야함(원본 데이터 texts)
vecotrdb._collection.count()

540

### 크로마(Chroma) DB에서 데이터 불러들이기

In [9]:
"""
<처리 순서>
 1. 임베딩 모델 클래스 정의 - 저장시 사용한 전용 모델(nomic-embed-text) 사용
 2. 크로마 DB가 위치한 디렉토리 위치(이름)정의
 3. 크로마 클래스 생성하여 1~2번 포함하여 생성
"""

'\n<처리 순서>\n 1. 임베딩 모델 클래스 정의 - 저장시 사용한 전용 모델(nomic-embed-text) 사용\n 2. 크로마 DB가 위치한 디렉토리 위치(이름)정의\n 3. 크로마 클래스 생성하여 1~2번 포함하여 생성\n'

In [10]:
#  1. 임베딩 모델 클래스 정의 - 저장시 사용한 전용 모델(nomic-embed-text) 사용
embedding_function = OllamaEmbeddings(model="nomic-embed-text")

#  2. 크로마 DB가 위치한 디렉토리 위치(이름)정의
chroma_path = "./Chroma_DB/"

#  3. 크로마 클래스 생성하여 1~2번 포함하여 생성
vecotrdb = Chroma(
    # 임베딩 모델 정의
    embedding_function = embedding_function,

    # 불러들일 크로마 DB 폴더 위치 정의
    persist_directory = chroma_path
)

print("=======[Chroma DB에서 데이터 불러들이기 종료]======")

=======[Chroma DB에서 데이터 불러들이기 종료]======


In [11]:
### 불러들인 데이터(청크=문서)의 갯수 확인하기
vecotrdb._collection.count()

540

## 크로마(Chroma)DB에서 불러들인 데이터를 이용하여 검색(search)

In [12]:
"""
<벡터 데이터베이스에서 검색 방법>
 * 벡터 데이터베이스는 클래스로 검색 함수를 제공하고 있음
 * 검색 함수
   - similarity_search(질문, k=2)
     : 단순히 유사성 검색을 통해 즉각적인 결과를 확인하고자 할 때 사용
       (랭체인 파이프라인에 적용안함, `단순 검색`으로 사용)
       (RAG 통합 서비스시에는 사용 못함, 테스트용으로 사용)
     : 질문 -> 사용자가 요청에 사용할 입력 받은 문장을 의미함
     : k   -> 질문과 유사한 조회 갯수(조회 청크(문서) 갯수)


   - as_retriver(search_kwargs={"k" : n})
     : 랭체인(lang chain) 파이프라인(랭체인) 연결을 통해 **"RAG 통합 서비스시에 사용"** 됨
     : 랭체인 파이프라인(랭체인)으로 연결시 사용하는 "표준 함수"임
     : search_kwargs={"k" : n}
       -> 질문과 유사한 조회 갯수(조회 청크 갯수)
       -> 질문은 파이프라인(랭체인)에서 연결되어 사용되기에 별도로 검색 함수에 추가하지 않음
"""


'\n<벡터 데이터베이스에서 검색 방법>\n * 벡터 데이터베이스는 클래스로 검색 함수를 제공하고 있음\n * 검색 함수\n   - similarity_search(질문, k=2)\n     : 단순히 유사성 검색을 통해 즉각적인 결과를 확인하고자 할 때 사용\n       (랭체인 파이프라인에 적용안함, `단순 검색`으로 사용)\n       (RAG 통합 서비스시에는 사용 못함, 테스트용으로 사용)\n     : 질문 -> 사용자가 요청에 사용할 입력 받은 문장을 의미함\n     : k   -> 질문과 유사한 조회 갯수(조회 청크(문서) 갯수)\n\n\n   - as_retriver(search_kwargs={"k" : n})\n     : 랭체인(lang chain) 파이프라인(랭체인) 연결을 통해 **"RAG 통합 서비스시에 사용"** 됨\n     : 랭체인 파이프라인(랭체인)으로 연결시 사용하는 "표준 함수"임\n     : search_kwargs={"k" : n}\n       -> 질문과 유사한 조회 갯수(조회 청크 갯수)\n       -> 질문은 파이프라인(랭체인)에서 연결되어 사용되기에 별도로 검색 함수에 추가하지 않음\n'

### similarity_search() 검색 함수 사용

In [13]:
"""
<처리 순서>
 1. 예시 질문 정의 : 입력 받아도 되며, 문자열로 정의해도 됨(테스트용)
 2. 검색 함수 정의 : 정의시 응답 갯수 정의 및 검색 결과 반환함
 3. 검색 결과 출력 : 응답 갯수가 여러개인 경우 반복문을 이용하여 응답 결과 출력 처리
"""


'\n<처리 순서>\n 1. 예시 질문 정의 : 입력 받아도 되며, 문자열로 정의해도 됨(테스트용)\n 2. 검색 함수 정의 : 정의시 응답 갯수 정의 및 검색 결과 반환함\n 3. 검색 결과 출력 : 응답 갯수가 여러개인 경우 반복문을 이용하여 응답 결과 출력 처리\n'

In [26]:
###  1. 예시 질문 정의 : 입력 받아도 되며, 문자열로 정의해도 됨(테스트용)
question = "수도권 주택 매매 전망"

###  2. 검색 함수 정의 : 정의시 응답 갯수 정의 및 검색 결과 반환함
# - 응답 갯수를 정의 해야함
# - 응답 결과 타임 : 리스트(list) 타입
# - 반환 데이터 형태 : 청크(문서-Document)
top_docs = vecotrdb.similarity_search(question, k=3)
top_docs

###  3. 검색 결과 출력 : 응답 갯수가 여러개인 경우 반복문을 이용하여 응답 결과 출력 처리
# 본문 내용 모두 각각 출력(print)

# - for 문 인덱스 버전
for i in range(0, len(top_docs), 1):
    content = top_docs[i].page_content
    print(f"[본문 내용] : {content}")
    print("="*100)

# - for 문 리스트에서 추출
for i in top_docs:
    content = i.page_content
    print(f"[본문 내용] : {content}")
    print("="*100)

for i, doc in enumerate(top_docs):
    print(f"[본문내용{i+1}번째]:{doc.page_content}")

[본문 내용] : 아파트 전세 수요는 지속될 것으로 예상된다. 반면 분양물량, 착공물량, 주택 인허가물량이 모두 감소하
며 향후 입주물량 감소에 대한 우려가 부각되고 있어 아파트로 집중된 전세 수요를 충족시킬 수 있을
지 의문이다. 
한편 비수도권에서도 아파트 전세 수요 집중 현상은 지속될 수 있으나 수도권에 비해 매수 부담이 적
어 전세로 전환되는 수요가 크지 않을 수 있다. 또한 입주물량이 큰 폭으로 감소하지 않을 것으로 예상
되면서 전세가격 상승세는 다소 제한적일 것으로 보인다.
[본문 내용] : 아파트 전세 수요는 지속될 것으로 예상된다. 반면 분양물량, 착공물량, 주택 인허가물량이 모두 감소하
며 향후 입주물량 감소에 대한 우려가 부각되고 있어 아파트로 집중된 전세 수요를 충족시킬 수 있을
지 의문이다. 
한편 비수도권에서도 아파트 전세 수요 집중 현상은 지속될 수 있으나 수도권에 비해 매수 부담이 적
어 전세로 전환되는 수요가 크지 않을 수 있다. 또한 입주물량이 큰 폭으로 감소하지 않을 것으로 예상
되면서 전세가격 상승세는 다소 제한적일 것으로 보인다.
[본문 내용] : 아파트 전세 수요는 지속될 것으로 예상된다. 반면 분양물량, 착공물량, 주택 인허가물량이 모두 감소하
며 향후 입주물량 감소에 대한 우려가 부각되고 있어 아파트로 집중된 전세 수요를 충족시킬 수 있을
지 의문이다. 
한편 비수도권에서도 아파트 전세 수요 집중 현상은 지속될 수 있으나 수도권에 비해 매수 부담이 적
어 전세로 전환되는 수요가 크지 않을 수 있다. 또한 입주물량이 큰 폭으로 감소하지 않을 것으로 예상
되면서 전세가격 상승세는 다소 제한적일 것으로 보인다.
[본문 내용] : 아파트 전세 수요는 지속될 것으로 예상된다. 반면 분양물량, 착공물량, 주택 인허가물량이 모두 감소하
며 향후 입주물량 감소에 대한 우려가 부각되고 있어 아파트로 집중된 전세 수요를 충족시킬 수 있을
지 의문이다. 
한편 비수도권에서도 아파트 전세 수요 집중 현상은 지속될 수 있으나 수도권에 비해 매

In [15]:
"""
[실습]
* 1. 어떠한 PDF 파일을 넣어도 로드 및 분할 처리하는 함수 정의
* 2. 1번의 분할 처리 함수를 통한 데이터를 이용해서 Chroma DB에 저장 하는 함수 정의
      - 저장 디렉토리명: Chroma_DB_exe
* 3. 2번에서 저장된 Chroma DB의 데이터를 불러들이는 함수 정의
* 4. 3번에서 불러들인 데이터를 이용하여,
     - 질문과 답변을 계속해서 진행할 수 있도록 처리
"""

'\n[실습]\n* 1. 어떠한 PDF 파일을 넣어도 로드 및 분할 처리하는 함수 정의\n* 2. 1번의 분할 처리 함수를 통한 데이터를 이용해서 Chroma DB에 저장 하는 함수 정의\n      - 저장 디렉토리명: Chroma_DB_exe\n* 3. 2번에서 저장된 Chroma DB의 데이터를 불러들이는 함수 정의\n* 4. 3번에서 불러들인 데이터를 이용하여,\n     - 질문과 답변을 계속해서 진행할 수 있도록 처리\n'

In [16]:
def load_split(pdf):
    file_path = "./data/" + pdf
    # 로더 클래스 생성
    loader = PyPDFLoader(file_path)
    # 파일 불러오기
    pages = loader.load()
    # 텍스트 분할기 클래스 생성
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 200
    )
    # 분할하기
    texts = text_splitter.split_documents(pages)
    # 분할한 텍스트 반환
    return texts

In [17]:
### 크로마 DB 저장 함수
def chroma_save(p_texts):
    # 임베딩 클래스 정의
    embedding_function = OllamaEmbeddings(model="nomic-embed-text")

    # 크로마 파일 위치 저장
    chroma_save_path = "./Chroma_DB_exe"

    # 크로마 DB 클래스 생성 및 저장 처리
    vecotrdb = Chroma.from_documents(
        # 원본 데이터
        documents = p_texts,

        # 문자를 숫자로 변환
        embedding = embedding_function,

        # 저장 위치 폴더
        persist_directory = chroma_save_path
    )
    print(f"저장한 데이터의 갯수 : {vecotrdb._collection.count()}")
    return vecotrdb

In [18]:
### 크로마 DB에서 데이터 불러들이기

def chroma_read():
    # 임베딩 클래스 정의
    embedding_function = OllamaEmbeddings(model="nomic-embed-text")

    # 크로마 DB가 위치한 디렉토리 정의
    chroma_path = "./Chroma_DB_exe"

    # 임베딩 클래스 생성 및 위치한 디렉토리 정의
    vecotrdb = Chroma(
        # 임베딩 모델 정의
        embedding_function = embedding_function,

        # 불러들일 크로마 DB 폴더 위치 정의
        persist_directory = chroma_path
    )
    print(f"불러들인 데이터의 갯수 : {vecotrdb._collection.count()}")
    return vecotrdb

In [24]:
def search(p_vecotrdb):
    while True:
        ### 입력받을 질문
        question = input("질문을 입력하세요")
        if question == "quit":
            break
        ### 검색 함수 정의
        top_docs = p_vecotrdb.similarity_search(question, k=5)

        for i, doc in enumerate(top_docs):
            print(f"[본문내용{i+1}번째]:{doc.page_content}")

In [23]:
a = load_split("2024_KB_부동산_보고서_최종.pdf")
b = chroma_save(a)
c = chroma_read()
d = search(c)

KeyboardInterrupt: 

In [21]:
### 클래스 생성시 self는 모든 함수마다 무조건 넣어준다고 생각

class Pdf_To_Search:
    def __init__(self, pdf):
        self.pdf = pdf

    # 로드 및 분리 함수
    def load_split(self):
        file_path = "./data/" + self.pdf
        # 로더 클래스 생성
        loader = PyPDFLoader(file_path)
        # 파일 불러오기
        pages = loader.load()
        # 텍스트 분할기 클래스 생성
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size = 1000,
            chunk_overlap = 200
        )
        # 분할하기
        self.texts = text_splitter.split_documents(pages)
        

    ### 크로마 DB 저장 함수
    def chroma_save(self):
        # 임베딩 클래스 정의
        embedding_function = OllamaEmbeddings(model="nomic-embed-text")

        # 크로마 파일 위치 저장
        chroma_save_path = "./Chroma_DB_exe"

        # 크로마 DB 클래스 생성 및 저장 처리
        self.vecotrdb = Chroma.from_documents(
            # 원본 데이터
            documents = self.texts,

            # 문자를 숫자로 변환
            embedding = embedding_function,

            # 저장 위치 폴더
            persist_directory = chroma_save_path
        )
        print(f"저장한 데이터의 갯수 : {self.vecotrdb._collection.count()}")
        return self.vecotrdb

    ### 크로마 DB에서 데이터 불러들이기

    def chroma_read(self):
        # 임베딩 클래스 정의
        embedding_function = OllamaEmbeddings(model="nomic-embed-text")

        # 크로마 DB가 위치한 디렉토리 정의
        chroma_path = "./Chroma_DB_exe"

        # 임베딩 클래스 생성 및 위치한 디렉토리 정의
        self.vecotrdb = Chroma(
            # 임베딩 모델 정의
            embedding_function = embedding_function,

            # 불러들일 크로마 DB 폴더 위치 정의
            persist_directory = chroma_path
        )
        print(f"불러들인 데이터의 갯수 : {self.vecotrdb._collection.count()}")
        return self.vecotrdb
    def search(self):
        while True:
            ### 입력받을 질문
            question = input("질문을 입력하세요")
            if question == "quit":
                break
            ### 검색 함수 정의
            top_docs = self.vecotrdb.similarity_search(question, k=5)

            for i, doc in enumerate(top_docs):
                print(f"[본문내용{i+1}번째]:{doc.page_content}")

In [22]:
pts = Pdf_To_Search("2024_KB_부동산_보고서_최종.pdf")
pts.chroma_read()
pts.search()

불러들인 데이터의 갯수 : 270
[본문내용1번째]:18 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
그림Ⅰ-30. 수도권 입주물량과 전세가격 변동률 추이  그림Ⅰ-31. 기타지방 입주물량과 전세가격 변동률 추이 
 
 
 
자료: KB국민은행, 부동산114  자료: KB국민은행, 부동산114
[본문내용2번째]:18 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
그림Ⅰ-30. 수도권 입주물량과 전세가격 변동률 추이  그림Ⅰ-31. 기타지방 입주물량과 전세가격 변동률 추이 
 
 
 
자료: KB국민은행, 부동산114  자료: KB국민은행, 부동산114
[본문내용3번째]:38 
2024 KB 부동산 보고서: 주택시장 설문조사 
 
 
그림Ⅱ-17. 고자산가의 부동산 상담 및 자문 내용  그림Ⅱ-18. 고자산가의 처분 희망 보유 부동산 
 
 
 
자료: KB경영연구소  자료: KB경영연구소 
 
 
[참고] 설문조사 개요 
조사 대상: 부동산시장 전문가 172명, KB 협력 공인중개사 523명, KB자산관리전문가(PB) 73명 
조사 방법: 온라인, 이메일, 팩스 등을 통한 설문조사 
조사 기간: 2024 년 1 월 2 일~12일 
조사 내용: 2024 년 주택 매매(전세)시장 전망, 주택 경기 회복 정책, 투자 유망 부동산, PB 고객 상담 및 자문 현황 등
[본문내용4번째]:38 
2024 KB 부동산 보고서: 주택시장 설문조사 
 
 
그림Ⅱ-17. 고자산가의 부동산 상담 및 자문 내용  그림Ⅱ-18. 고자산가의 처분 희망 보유 부동산 
 
 
 
자료: KB경영연구소  자료: KB경영연구소 
 
 
[참고] 설문조사 개요 
조사 대상: 부동산시장 전문가 172명, KB 협력 공인중개사 523명, KB자산관리전문가(PB) 73명 
조사 방법: 온라인, 이메일, 팩스 등을 통한 설문조사 
조사 기간: 2024 년 1 월 2 일~12일 
조사 내용: 2024 년 주택 매매(전세)시장 전망, 주택 경기 회복 정책,